# HERE Berlin School-Zone Sign Pipeline

This notebook implements a school-region extraction workflow using `sample_school_contexts.geojson` as the input source.

1. Load the sample school context, road, and school geometries.
2. Call Gemini through LangChain using the API key from `.env`.
3. Select the school-related road LineStrings and infer the school region.
4. Export QGIS-ready GeoJSON files for mapping and review.

The notebook is defensive: it can still be inspected before your Gemini key is available.

## 1. Set Up Notebook Environment and Dependencies

In [1]:
from __future__ import annotations

import json
import math
import os
import random
import re
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_google_genai import ChatGoogleGenerativeAI
except Exception:
    ChatPromptTemplate = None
    ChatGoogleGenerativeAI = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from shapely.geometry import LineString, Point, Polygon, mapping
except Exception:
    LineString = None
    Point = None
    Polygon = None
    mapping = None

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print(f"Python: {os.sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Shapely available: {LineString is not None}")
print(f"Matplotlib available: {plt is not None}")
print(f"LangChain available: {ChatPromptTemplate is not None and ChatGoogleGenerativeAI is not None}")
print(f"Dotenv available: {load_dotenv is not None}")

k:\Tech\Hackathon\Here-Berlin-2026\Sign_Detection\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Python: 3.14.2
Pandas: 3.0.2
Shapely available: True
Matplotlib available: True
LangChain available: True
Dotenv available: True


## 2. Define Configuration and Constants

In [2]:
@dataclass
class PipelineConfig:
    root_dir: Path
    data_dir: Path
    cropped_dir: Path
    output_dir: Path

    mapillary_csv: Path
    schoolzone_geojson: Path
    remaining_geojson: Path
    roads_geojson: Path
    schools_geojson: Path
    sample_school_contexts_geojson: Path
    school_region_output_dir: Path
    school_region_geojson: Path
    school_region_response_json: Path

    use_mock_mode: bool = True
    gemini_model: str = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")
    max_sign_match_m: float = 80.0
    max_school_match_m: float = 500.0
    max_road_snap_m: float = 50.0
    default_zone_cap_m: float = 500.0
    line_buffer_m: float = 15.0

    image_detections_jsonl: Path = Path("outputs/image_detections.jsonl")
    validated_decisions_jsonl: Path = Path("outputs/validated_zone_decisions.jsonl")
    merged_detections_geojson: Path = Path("outputs/merged_detections.geojson")
    inferred_lines_geojson: Path = Path("outputs/inferred_zone_extents_lines.geojson")
    inferred_polygons_geojson: Path = Path("outputs/inferred_zone_extents_polygons.geojson")


def infer_workspace_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "mapillary_image_metadata.csv").exists():
        return cwd
    if (cwd.parent / "mapillary_image_metadata.csv").exists():
        return cwd.parent
    return cwd


ROOT = infer_workspace_root()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = PipelineConfig(
    root_dir=ROOT,
    data_dir=ROOT,
    cropped_dir=ROOT / "cropped_image",
    output_dir=OUTPUT_DIR,
    mapillary_csv=ROOT / "mapillary_image_metadata.csv",
    schoolzone_geojson=ROOT / "PotsdamBerlin_schoolzone_speedlimit_signs.geojson",
    remaining_geojson=ROOT / "PotsdamBerlin_remaining_speed_limit_signs.geojson",
    roads_geojson=ROOT / "PotsdamBerlin_road_geometry.geojson",
    schools_geojson=ROOT / "PotsdamBerlin_school_pointsofinterest.geojson",
    sample_school_contexts_geojson=ROOT / "sample_school_contexts.geojson",
    school_region_output_dir=OUTPUT_DIR / "school_region",
    school_region_geojson=OUTPUT_DIR / "school_region" / "school_related_roads.geojson",
    school_region_response_json=OUTPUT_DIR / "school_region" / "gemini_school_region_response.json",
    image_detections_jsonl=OUTPUT_DIR / "image_detections.jsonl",
    validated_decisions_jsonl=OUTPUT_DIR / "validated_zone_decisions.jsonl",
    merged_detections_geojson=OUTPUT_DIR / "merged_detections.geojson",
    inferred_lines_geojson=OUTPUT_DIR / "inferred_zone_extents_lines.geojson",
    inferred_polygons_geojson=OUTPUT_DIR / "inferred_zone_extents_polygons.geojson",
)

CONFIG.school_region_output_dir.mkdir(parents=True, exist_ok=True)

print(asdict(CONFIG))

{'root_dir': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection'), 'data_dir': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection'), 'cropped_dir': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/cropped_image'), 'output_dir': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/outputs'), 'mapillary_csv': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/mapillary_image_metadata.csv'), 'schoolzone_geojson': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/PotsdamBerlin_schoolzone_speedlimit_signs.geojson'), 'remaining_geojson': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/PotsdamBerlin_remaining_speed_limit_signs.geojson'), 'roads_geojson': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/PotsdamBerlin_road_geometry.geojson'), 'schools_geojson': WindowsPath('K:/Tech/Hackathon/Here-Berlin-2026/Sign_Detection/PotsdamBerlin_school_pointsofinterest.geojson'), 'sample_school_contexts

In [7]:
if load_dotenv is not None:
    load_dotenv(ROOT / ".env", override=False)


def read_geojson(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def get_gemini_api_key() -> str:
    for env_name in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        value = os.getenv(env_name)
        if value:
            return value
    raise EnvironmentError(
        "No Gemini API key found. Set GOOGLE_API_KEY or GEMINI_API_KEY in your .env file.",
    )


def summarize_feature(feature: Dict[str, Any], index: int) -> Dict[str, Any]:
    geometry = feature.get("geometry", {})
    properties = feature.get("properties", {})
    summary = {
        "index": index,
        "feature_type": properties.get("feature_type"),
        "source": properties.get("source"),
        "road_name": properties.get("road_name"),
        "travel_direction": properties.get("travel_direction"),
        "average_speed": properties.get("average_speed"),
        "schoolzone_gfrgroup": properties.get("schoolzone_gfrgroup"),
        "speedlimit_gfrgroup": properties.get("speedlimit_gfrgroup"),
        "geometry_type": geometry.get("type"),
    }
    if geometry.get("type") == "Point":
        summary["coordinates"] = geometry.get("coordinates")
    elif geometry.get("type") == "LineString":
        coords = geometry.get("coordinates", [])
        summary["coordinates"] = {
            "start": coords[0] if coords else None,
            "end": coords[-1] if coords else None,
            "point_count": len(coords),
        }
    else:
        summary["coordinates"] = geometry.get("coordinates")
    return summary


def build_feature_catalog(input_geojson: Dict[str, Any]) -> List[Dict[str, Any]]:
    features = input_geojson.get("features", [])
    return [summarize_feature(feature, index) for index, feature in enumerate(features)]


def build_school_region_prompt(input_geojson: Dict[str, Any]) -> str:
    catalog = build_feature_catalog(input_geojson)
    return f"""You are a senior geospatial analyst working on a school-zone extraction task for QGIS.

Goal: identify the school region, the exact school point, and every road LineString that is part of the school-related corridor around the school.

Input: a GeoJSON FeatureCollection named `sample_school_contexts.geojson`. It contains a school feature, road features, and possible speed-limit sign features.

Instructions:
1. Determine which feature is the school point.
2. Select all road LineString features that belong to the school-related path or immediate school region.
3. Prefer roads that visually or semantically connect to the school area, especially streets carrying school-zone speed-limit context.
4. Return only JSON. No markdown, no explanation outside JSON.
5. Keep coordinates in [longitude, latitude] order exactly as in GeoJSON.
6. For the region geometry, return a polygon that encloses the school point and the selected road corridor. If you cannot infer a precise polygon, return null and the notebook will create a safe fallback polygon.

Return this JSON schema exactly:
{{
  \"school_label\": string,
  \"school_feature_index\": integer | null,
  \"selected_road_feature_indices\": [integer, ...],
  \"selected_supporting_feature_indices\": [integer, ...],
  \"school_region_polygon\": {{
    \"type\": \"Polygon\",
    \"coordinates\": [[[lon, lat], ...]]
  }} | null,
  \"confidence\": number,
  \"reasoning\": string,
  \"notes\": [string, ...]
}}

Important constraints:
- Every selected road feature must already exist in the input FeatureCollection.
- Include only LineString roads in selected_road_feature_indices.
- selected_supporting_feature_indices can include the school point or sign-like features.
- Confidence must be between 0 and 1.
- The output is intended to be rendered in QGIS, so favor compact but complete geometry selection.

Input feature catalog:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
"""


def normalize_llm_text(content: Any) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: List[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def extract_json_object(text: str) -> Dict[str, Any]:
    candidate = text.strip()
    if candidate.startswith("```"):
        candidate = re.sub(r"^```(?:json)?\s*", "", candidate, flags=re.IGNORECASE)
        candidate = re.sub(r"\s*```$", "", candidate)
    start = candidate.find("{")
    end = candidate.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"Could not locate JSON object in model response: {text[:400]}")
    return json.loads(candidate[start : end + 1])


def fallback_region_polygon(input_geojson: Dict[str, Any], selected_indices: List[int], school_index: Optional[int]) -> Optional[Dict[str, Any]]:
    coordinates: List[Tuple[float, float]] = []
    features = input_geojson.get("features", [])
    interesting_indices = list(dict.fromkeys([*(selected_indices or []), school_index] if school_index is not None else selected_indices or []))
    for index in interesting_indices:
        if index is None or index < 0 or index >= len(features):
            continue
        geometry = features[index].get("geometry", {})
        geom_type = geometry.get("type")
        geom_coords = geometry.get("coordinates", [])
        if geom_type == "Point" and len(geom_coords) >= 2:
            coordinates.append((float(geom_coords[0]), float(geom_coords[1])))
        elif geom_type == "LineString":
            for lon, lat in geom_coords:
                coordinates.append((float(lon), float(lat)))
    if not coordinates:
        return None
    min_lon = min(lon for lon, _ in coordinates)
    max_lon = max(lon for lon, _ in coordinates)
    min_lat = min(lat for _, lat in coordinates)
    max_lat = max(lat for _, lat in coordinates)
    pad = 0.00045
    return {
        "type": "Polygon",
        "coordinates": [[
            [min_lon - pad, min_lat - pad],
            [max_lon + pad, min_lat - pad],
            [max_lon + pad, max_lat + pad],
            [min_lon - pad, max_lat + pad],
            [min_lon - pad, min_lat - pad],
        ]],
    }


def build_school_region_feature_collection(
    input_geojson: Dict[str, Any],
    llm_result: Dict[str, Any],
    model_name: str,
    source_path: Path,
) -> Dict[str, Any]:
    features = input_geojson.get("features", [])
    selected_road_indices = [int(index) for index in llm_result.get("selected_road_feature_indices", []) if isinstance(index, int) or str(index).isdigit()]
    selected_supporting_indices = [int(index) for index in llm_result.get("selected_supporting_feature_indices", []) if isinstance(index, int) or str(index).isdigit()]
    school_index = llm_result.get("school_feature_index")
    if isinstance(school_index, str) and school_index.isdigit():
        school_index = int(school_index)
    if not isinstance(school_index, int):
        school_index = None

    region_polygon = llm_result.get("school_region_polygon")
    if not region_polygon:
        region_polygon = fallback_region_polygon(input_geojson, selected_road_indices, school_index)

    output_features: List[Dict[str, Any]] = []
    source_indices: List[int] = []

    if school_index is not None and 0 <= school_index < len(features):
        school_feature = json.loads(json.dumps(features[school_index]))
        school_feature.setdefault("properties", {})
        school_feature["properties"].update({
            "output_role": "school_point",
            "source_feature_index": school_index,
        })
        output_features.append(school_feature)
        source_indices.append(school_index)

    for road_index in selected_road_indices:
        if road_index in source_indices:
            continue
        if road_index < 0 or road_index >= len(features):
            continue
        feature = features[road_index]
        if feature.get("geometry", {}).get("type") != "LineString":
            continue
        road_feature = json.loads(json.dumps(feature))
        road_feature.setdefault("properties", {})
        road_feature["properties"].update({
            "output_role": "school_related_road",
            "source_feature_index": road_index,
        })
        output_features.append(road_feature)
        source_indices.append(road_index)

    for support_index in selected_supporting_indices:
        if support_index in source_indices:
            continue
        if support_index < 0 or support_index >= len(features):
            continue
        feature = json.loads(json.dumps(features[support_index]))
        feature.setdefault("properties", {})
        feature["properties"].update({
            "output_role": "supporting_feature",
            "source_feature_index": support_index,
        })
        output_features.append(feature)
        source_indices.append(support_index)

    if region_polygon is not None:
        output_features.append({
            "type": "Feature",
            "geometry": region_polygon,
            "properties": {
                "output_role": "inferred_school_region",
                "source_feature_index": None,
            },
        })

    return {
        "type": "FeatureCollection",
        "features": output_features,
        "metadata": {
            "generated_by": "Gemini via LangChain",
            "model": model_name,
            "source_path": str(source_path),
            "selected_feature_indices": source_indices,
            "input_feature_count": len(features),
            "confidence": llm_result.get("confidence"),
            "school_label": llm_result.get("school_label"),
            "reasoning": llm_result.get("reasoning"),
            "notes": llm_result.get("notes", []),
        },
    }


def run_gemini_school_region_pipeline(config: PipelineConfig) -> Dict[str, Any]:
    if ChatPromptTemplate is None or ChatGoogleGenerativeAI is None:
        raise ImportError(
            "LangChain Gemini dependencies are missing. Install langchain and langchain-google-genai first.",
        )
    api_key = get_gemini_api_key()
    input_geojson = read_geojson(config.sample_school_contexts_geojson)
    prompt_text = build_school_region_prompt(input_geojson)
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "You are a precise geospatial extraction assistant. Return only strict JSON with the requested schema.",
        ),
        (
            "user",
            "{prompt_text}",
        ),
    ])
    llm = ChatGoogleGenerativeAI(
        model=config.gemini_model,
        temperature=0.0,
        google_api_key=api_key,
    )
    chain = prompt | llm
    response = chain.invoke({"prompt_text": prompt_text})
    response_text = normalize_llm_text(getattr(response, "content", response))
    llm_result = extract_json_object(response_text)

    feature_collection = build_school_region_feature_collection(
        input_geojson=input_geojson,
        llm_result=llm_result,
        model_name=config.gemini_model,
        source_path=config.sample_school_contexts_geojson,
    )

    config.school_region_output_dir.mkdir(parents=True, exist_ok=True)
    with config.school_region_geojson.open("w", encoding="utf-8") as handle:
        json.dump(feature_collection, handle, ensure_ascii=True, indent=2)
    with config.school_region_response_json.open("w", encoding="utf-8") as handle:
        json.dump(llm_result, handle, ensure_ascii=True, indent=2)

    return {
        "input_path": str(config.sample_school_contexts_geojson),
        "output_geojson": str(config.school_region_geojson),
        "output_response": str(config.school_region_response_json),
        "selected_road_count": len(llm_result.get("selected_road_feature_indices", [])),
        "selected_supporting_count": len(llm_result.get("selected_supporting_feature_indices", [])),
        "confidence": llm_result.get("confidence"),
        "school_label": llm_result.get("school_label"),
    }


print("Gemini school-region helpers are ready.")

Gemini school-region helpers are ready.


## 3. Create Core Data Structures

In [11]:
@dataclass
class TimeModifier:
    has_modifier: bool
    type: str
    value: str


@dataclass
class DetectionRecord:
    image_path: str
    image_id: Optional[str]
    latitude: Optional[float]
    longitude: Optional[float]
    sign_type: str
    speed_limit_kmh: Optional[int]
    time_modifier: TimeModifier
    confidence: float
    notes: str
    model_name: str


@dataclass
class MatchedDetection:
    detection: DetectionRecord
    matched_sign_index: Optional[int]
    matched_sign_type: Optional[str]
    sign_distance_m: Optional[float]
    matched_road_id: Optional[str]
    road_snap_distance_m: Optional[float]
    road_position_t: Optional[float]
    nearest_school_distance_m: Optional[float]
    nearest_school_suppliers: Optional[str]
    zone_role: str = "unknown"

## 4. Implement Main Processing Functions

In [12]:
def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Compute great-circle distance in meters."""
    r = 6371000.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def extract_speed_limit_kmh(value: Optional[str]) -> Optional[int]:
    if not value:
        return None
    match = re.search(r"V(\d+)", value)
    return int(match.group(1)) if match else None


def classify_sign_type(
    speed_group: Optional[str],
    schoolzone_group: Optional[str],
    suppliers: Optional[str] = None,
) -> str:
    text = " ".join([s for s in [speed_group, schoolzone_group, suppliers] if s]).lower()
    if "kindergarten" in text:
        return "Kindergarten"
    if "school" in text:
        return "School"
    return "General"


def parse_time_modifier(supplemental_group: Optional[str]) -> TimeModifier:
    if supplemental_group is None:
        return TimeModifier(False, "none", "")

    txt = supplemental_group.lower()
    if "timemodifier" in txt:
        return TimeModifier(True, "time_range", supplemental_group)
    if "daymodifier" in txt:
        return TimeModifier(True, "day_of_week", supplemental_group)
    if "school" in txt:
        return TimeModifier(True, "school_days", supplemental_group)
    if "rain" in txt or "snow" in txt:
        return TimeModifier(True, "weather", supplemental_group)
    return TimeModifier(True, "other", supplemental_group)


def road_polyline_length_m(coords: List[List[float]]) -> float:
    total = 0.0
    for i in range(len(coords) - 1):
        lon1, lat1 = coords[i]
        lon2, lat2 = coords[i + 1]
        total += haversine_m(lat1, lon1, lat2, lon2)
    return total


def project_point_to_segment_t(px: float, py: float, ax: float, ay: float, bx: float, by: float) -> Tuple[float, float, float]:
    vx = bx - ax
    vy = by - ay
    seg_len_sq = vx * vx + vy * vy
    if seg_len_sq == 0:
        return 0.0, ax, ay
    t = ((px - ax) * vx + (py - ay) * vy) / seg_len_sq
    t = max(0.0, min(1.0, t))
    return t, ax + t * vx, ay + t * vy


def to_local_xy_m(lat: float, lon: float, lat0: float, lon0: float) -> Tuple[float, float]:
    # Equirectangular approximation for short local distances.
    m_per_deg_lat = 111320.0
    m_per_deg_lon = 111320.0 * math.cos(math.radians(lat0))
    x = (lon - lon0) * m_per_deg_lon
    y = (lat - lat0) * m_per_deg_lat
    return x, y

In [13]:
def load_geojson_features(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    return payload.get("features", [])


def load_inputs(config: PipelineConfig) -> Dict[str, Any]:
    metadata_df = pd.read_csv(config.mapillary_csv)
    schoolzone_features = load_geojson_features(config.schoolzone_geojson)
    remaining_features = load_geojson_features(config.remaining_geojson)
    roads_features = load_geojson_features(config.roads_geojson)
    school_features = load_geojson_features(config.schools_geojson)

    return {
        "metadata_df": metadata_df,
        "schoolzone_features": schoolzone_features,
        "remaining_features": remaining_features,
        "roads_features": roads_features,
        "school_features": school_features,
    }


def discover_cropped_images(cropped_dir: Path) -> List[Path]:
    if not cropped_dir.exists():
        return []
    image_paths: List[Path] = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
        image_paths.extend(cropped_dir.glob(ext))
    return sorted(image_paths)


def parse_image_id_from_filename(image_path: Path) -> Optional[str]:
    tokens = re.findall(r"\d{6,}", image_path.stem)
    return tokens[0] if tokens else None


def find_metadata_row(metadata_df: pd.DataFrame, image_id: Optional[str]) -> Optional[pd.Series]:
    if image_id is None:
        return None
    found = metadata_df[metadata_df["image_id"].astype(str) == str(image_id)]
    if found.empty:
        return None
    return found.iloc[0]


def make_feature_collection(features: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {"type": "FeatureCollection", "features": features}

In [14]:
def mock_vision_inference(
    image_path: Path,
    metadata_row: Optional[pd.Series],
    config: PipelineConfig,
) -> DetectionRecord:
    image_id = parse_image_id_from_filename(image_path)
    lat = float(metadata_row["latitude"]) if metadata_row is not None else None
    lon = float(metadata_row["longitude"]) if metadata_row is not None else None

    seed_value = int(re.sub(r"\D", "", image_path.stem)[:8] or "42")
    rng = random.Random(seed_value)

    speed_limit = rng.choice([10, 20, 25, 30])
    sign_type = rng.choice(["School", "Kindergarten", "General"])
    has_mod = rng.random() < 0.35
    modifier = TimeModifier(
        has_modifier=has_mod,
        type=rng.choice(["time_range", "day_of_week", "none"]) if has_mod else "none",
        value=rng.choice(["07:00-17:00", "Mo-Fr", "Schultage", ""]) if has_mod else "",
    )

    return DetectionRecord(
        image_path=str(image_path),
        image_id=image_id,
        latitude=lat,
        longitude=lon,
        sign_type=sign_type,
        speed_limit_kmh=speed_limit,
        time_modifier=modifier,
        confidence=round(rng.uniform(0.65, 0.97), 3),
        notes="Mock inference result.",
        model_name="mock-detector-v1",
    )


def gemini_vision_inference_placeholder(
    image_path: Path,
    metadata_row: Optional[pd.Series],
    config: PipelineConfig,
) -> DetectionRecord:
    """Placeholder for Gemini API integration.

    TODO:
    - Read API key from GEMINI_API_KEY.
    - Send image + schema prompt.
    - Parse JSON strictly and fallback to safe defaults if parsing fails.
    """
    rec = mock_vision_inference(image_path, metadata_row, config)
    rec.notes = "Gemini placeholder used; replace with real API call."
    rec.model_name = config.gemini_model
    return rec


def nearest_sign_match(
    det: DetectionRecord,
    all_sign_features: List[Dict[str, Any]],
    max_distance_m: float,
) -> Tuple[Optional[int], Optional[str], Optional[float]]:
    if det.latitude is None or det.longitude is None:
        return None, None, None

    best_idx = None
    best_dist = float("inf")
    best_type = None

    for idx, feat in enumerate(all_sign_features):
        props = feat.get("properties", {})
        geom = feat.get("geometry", {})
        coords = geom.get("coordinates", [None, None])
        lon, lat = coords[0], coords[1]
        if lat is None or lon is None:
            continue
        d = haversine_m(det.latitude, det.longitude, lat, lon)
        if d < best_dist:
            best_dist = d
            best_idx = idx
            best_type = props.get("speedlimit_gfrgroup")

    if best_idx is None or best_dist > max_distance_m:
        return None, None, None
    return best_idx, best_type, best_dist

In [15]:
def nearest_road_snap(
    lat: Optional[float],
    lon: Optional[float],
    roads_features: List[Dict[str, Any]],
    max_snap_m: float,
) -> Tuple[Optional[str], Optional[float], Optional[float], Optional[List[List[float]]]]:
    if lat is None or lon is None:
        return None, None, None, None

    best_road_id = None
    best_dist = float("inf")
    best_t = None
    best_coords = None

    for feat in roads_features:
        coords = feat.get("geometry", {}).get("coordinates", [])
        if len(coords) < 2:
            continue

        props = feat.get("properties", {})
        road_id = props.get("id")

        local = [to_local_xy_m(c[1], c[0], lat, lon) for c in coords]
        px, py = 0.0, 0.0

        best_seg_dist = float("inf")
        best_seg_t = 0.0
        cumulative = 0.0
        proj_m = 0.0
        total_m = road_polyline_length_m(coords)
        if total_m == 0:
            continue

        for i in range(len(local) - 1):
            ax, ay = local[i]
            bx, by = local[i + 1]
            seg_t, qx, qy = project_point_to_segment_t(px, py, ax, ay, bx, by)
            dist = math.hypot(px - qx, py - qy)
            seg_len = haversine_m(coords[i][1], coords[i][0], coords[i + 1][1], coords[i + 1][0])
            if dist < best_seg_dist:
                best_seg_dist = dist
                best_seg_t = (cumulative + seg_t * seg_len) / total_m
            cumulative += seg_len

        if best_seg_dist < best_dist:
            best_dist = best_seg_dist
            best_road_id = road_id
            best_t = best_seg_t
            best_coords = coords

    if best_road_id is None or best_dist > max_snap_m:
        return None, None, None, None
    return best_road_id, best_dist, best_t, best_coords


def nearest_school(
    lat: Optional[float],
    lon: Optional[float],
    school_features: List[Dict[str, Any]],
    max_distance_m: float,
) -> Tuple[Optional[float], Optional[str]]:
    if lat is None or lon is None:
        return None, None

    best_dist = float("inf")
    best_suppliers = None

    for feat in school_features:
        coords = feat.get("geometry", {}).get("coordinates", [None, None])
        props = feat.get("properties", {})
        slon, slat = coords[0], coords[1]
        if slat is None or slon is None:
            continue
        d = haversine_m(lat, lon, slat, slon)
        if d < best_dist:
            best_dist = d
            best_suppliers = props.get("suppliers")

    if best_dist > max_distance_m:
        return None, None
    return best_dist, best_suppliers

## 4.5 Multi-Step Evidence Validation

In [16]:
@dataclass
class ZoneEvidence:
    road_id: Optional[str]
    sign_type: str
    speed_limit_kmh: Optional[int]
    cluster_size: int
    image_count: int
    school_distance_m: Optional[float]
    road_span_m: Optional[float]
    start_position_t: Optional[float]
    end_position_t: Optional[float]
    has_conditional_modifier: bool
    conditional_modifier: str
    supporting_image_ids: List[str]
    supporting_notes: List[str]
    confidence: float


@dataclass
class ValidatedZoneDecision:
    final_sign_type: str
    speed_limit_kmh: Optional[int]
    time_modifier: TimeModifier
    is_school_zone: bool
    is_conditional: bool
    zone_start_m: Optional[float]
    zone_end_m: Optional[float]
    cluster_count: int
    image_count: int
    confidence: float
    notes: str


def collect_zone_evidence(
    detections: List[DetectionRecord],
    matched_rows: List[MatchedDetection],
    config: PipelineConfig,
) -> List[ZoneEvidence]:
    grouped: Dict[Tuple[Optional[str], Optional[int]], List[MatchedDetection]] = {}
    for row in matched_rows:
        key = (row.matched_road_id, row.detection.speed_limit_kmh)
        grouped.setdefault(key, []).append(row)

    evidence_items: List[ZoneEvidence] = []
    for (road_id, speed_limit), rows in grouped.items():
        if not rows:
            continue
        rows_sorted = sorted([r for r in rows if r.road_position_t is not None], key=lambda r: r.road_position_t or 0.0)
        if rows_sorted:
            start_t = rows_sorted[0].road_position_t
            end_t = rows_sorted[-1].road_position_t
            road_span_m = max(0.0, (end_t - start_t) * config.default_zone_cap_m) if start_t is not None and end_t is not None else None
        else:
            start_t = None
            end_t = None
            road_span_m = None

        supporting_image_ids = [r.detection.image_id for r in rows if r.detection.image_id]
        notes = []
        school_distances = [r.nearest_school_distance_m for r in rows if r.nearest_school_distance_m is not None]
        school_distance_m = min(school_distances) if school_distances else None

        conditional_signals = []
        sign_types = [r.detection.sign_type for r in rows if r.detection.sign_type]
        for r in rows:
            tm = r.detection.time_modifier
            if tm.has_modifier:
                conditional_signals.append(tm.value or tm.type)

        unique_images = len(set(supporting_image_ids))
        cluster_size = len(rows)
        confidence = round(min(0.99, 0.35 + 0.1 * cluster_size + 0.05 * unique_images), 3)

        if road_id is None:
            notes.append("No road match")
        if school_distance_m is None:
            notes.append("No nearby school within threshold")
        if conditional_signals:
            notes.append(f"Conditional signals: {', '.join(sorted(set(conditional_signals)))}")

        if any(s == "School" for s in sign_types):
            sign_type = "School"
        elif any(s == "Kindergarten" for s in sign_types):
            sign_type = "Kindergarten"
        else:
            sign_type = "General"

        evidence_items.append(
            ZoneEvidence(
                road_id=road_id,
                sign_type=sign_type,
                speed_limit_kmh=speed_limit,
                cluster_size=cluster_size,
                image_count=unique_images,
                school_distance_m=school_distance_m,
                road_span_m=road_span_m,
                start_position_t=start_t,
                end_position_t=end_t,
                has_conditional_modifier=bool(conditional_signals),
                conditional_modifier="; ".join(sorted(set(conditional_signals))) if conditional_signals else "",
                supporting_image_ids=supporting_image_ids,
                supporting_notes=notes,
                confidence=confidence,
            )
        )

    return evidence_items


def validate_zone_evidence(evidence: ZoneEvidence, config: PipelineConfig) -> ValidatedZoneDecision:
    is_school_zone = evidence.sign_type in {"School", "Kindergarten"} or (
        evidence.school_distance_m is not None and evidence.school_distance_m <= config.max_school_match_m
    )
    is_conditional = evidence.has_conditional_modifier
    final_sign_type = evidence.sign_type
    time_modifier = TimeModifier(
        has_modifier=is_conditional,
        type="time_range" if is_conditional and evidence.conditional_modifier else "none",
        value=evidence.conditional_modifier if is_conditional else "",
    )

    zone_start_m = None
    zone_end_m = None
    if evidence.start_position_t is not None:
        zone_start_m = max(0.0, evidence.start_position_t * config.default_zone_cap_m)
    if evidence.end_position_t is not None:
        zone_end_m = min(config.default_zone_cap_m, evidence.end_position_t * config.default_zone_cap_m)

    notes_parts = list(evidence.supporting_notes)
    if evidence.cluster_size < 2:
        notes_parts.append("Sparse cluster, final answer downgraded")
    if not is_school_zone:
        notes_parts.append("No school/classification evidence strong enough, keeping general classification")
    if is_conditional:
        notes_parts.append("Conditional restriction detected")

    confidence = evidence.confidence
    if evidence.cluster_size < 2:
        confidence -= 0.15
    if evidence.school_distance_m is None:
        confidence -= 0.15
    if is_conditional:
        confidence += 0.05
    confidence = max(0.0, min(1.0, round(confidence, 3)))

    return ValidatedZoneDecision(
        final_sign_type=final_sign_type,
        speed_limit_kmh=evidence.speed_limit_kmh,
        time_modifier=time_modifier,
        is_school_zone=is_school_zone,
        is_conditional=is_conditional,
        zone_start_m=zone_start_m,
        zone_end_m=zone_end_m,
        cluster_count=evidence.cluster_size,
        image_count=evidence.image_count,
        confidence=confidence,
        notes=" | ".join(notes_parts),
    )


def multi_step_validate_detections(
    detections: List[DetectionRecord],
    matched_rows: List[MatchedDetection],
    config: PipelineConfig,
) -> List[ValidatedZoneDecision]:
    evidence_items = collect_zone_evidence(detections, matched_rows, config)
    return [validate_zone_evidence(item, config) for item in evidence_items]

## 5. Add Input Validation and Error Handling

In [17]:
def validate_config(config: PipelineConfig) -> None:
    required = [
        config.mapillary_csv,
        config.schoolzone_geojson,
        config.remaining_geojson,
        config.roads_geojson,
        config.schools_geojson,
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Missing required files: {missing}")


def validate_detection(record: DetectionRecord) -> None:
    if record.sign_type not in {"Kindergarten", "School", "General"}:
        raise ValueError(f"Unsupported sign_type: {record.sign_type}")
    if not (0.0 <= record.confidence <= 1.0):
        raise ValueError(f"confidence out of range: {record.confidence}")
    if record.speed_limit_kmh is not None and record.speed_limit_kmh <= 0:
        raise ValueError(f"Invalid speed limit: {record.speed_limit_kmh}")


def safe_json_dump(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=True, indent=2)


validate_config(CONFIG)
print("Validation passed.")

Validation passed.


## 6. Build an Orchestration Function

In [18]:
def infer_zone_features(matched_rows: List[MatchedDetection], config: PipelineConfig) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    rows = [r for r in matched_rows if r.matched_road_id and r.road_position_t is not None]
    grouped: Dict[str, List[MatchedDetection]] = {}
    for row in rows:
        grouped.setdefault(row.matched_road_id, []).append(row)

    line_features: List[Dict[str, Any]] = []
    poly_features: List[Dict[str, Any]] = []

    for road_id, items in grouped.items():
        if len(items) < 2:
            continue

        sorted_items = sorted(items, key=lambda x: x.road_position_t or 0.0)
        start = sorted_items[0]
        end = sorted_items[-1]

        start_t = start.road_position_t or 0.0
        end_t = end.road_position_t or start_t
        if end_t <= start_t:
            continue

        start_det = start.detection
        end_det = end.detection
        speed_limit = start_det.speed_limit_kmh or end_det.speed_limit_kmh
        conf = round((start_det.confidence + end_det.confidence) / 2.0, 3)

        start_lon, start_lat = start_det.longitude, start_det.latitude
        end_lon, end_lat = end_det.longitude, end_det.latitude
        if None in [start_lon, start_lat, end_lon, end_lat]:
            continue

        line_coords = [[start_lon, start_lat], [end_lon, end_lat]]
        line_features.append(
            {
                "type": "Feature",
                "geometry": {"type": "LineString", "coordinates": line_coords},
                "properties": {
                    "road_id": road_id,
                    "zone_type": "school_zone_inferred",
                    "speed_limit_kmh": speed_limit,
                    "confidence": conf,
                },
            }
        )

        if LineString is not None:
            deg_buffer = config.line_buffer_m / 111320.0
            buffered = LineString(line_coords).buffer(deg_buffer)
            poly_geom = mapping(buffered)
        else:
            min_lon = min(start_lon, end_lon)
            max_lon = max(start_lon, end_lon)
            min_lat = min(start_lat, end_lat)
            max_lat = max(start_lat, end_lat)
            pad = config.line_buffer_m / 111320.0
            poly_geom = {
                "type": "Polygon",
                "coordinates": [[
                    [min_lon - pad, min_lat - pad],
                    [max_lon + pad, min_lat - pad],
                    [max_lon + pad, max_lat + pad],
                    [min_lon - pad, max_lat + pad],
                    [min_lon - pad, min_lat - pad],
                ]],
            }

        poly_features.append(
            {
                "type": "Feature",
                "geometry": poly_geom,
                "properties": {
                    "road_id": road_id,
                    "zone_type": "school_zone_inferred",
                    "speed_limit_kmh": speed_limit,
                    "confidence": conf,
                },
            }
        )

    return line_features, poly_features


def export_validated_decisions(path: Path, decisions: List[ValidatedZoneDecision]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for decision in decisions:
            payload = asdict(decision)
            payload["time_modifier"] = asdict(decision.time_modifier)
            f.write(json.dumps(payload, ensure_ascii=True) + "\n")


def run_pipeline(config: PipelineConfig) -> Dict[str, Any]:
    validate_config(config)
    data = load_inputs(config)

    metadata_df = data["metadata_df"]
    schoolzone_features = data["schoolzone_features"]
    remaining_features = data["remaining_features"]
    roads_features = data["roads_features"]
    school_features = data["school_features"]

    all_sign_features = schoolzone_features + remaining_features

    image_paths = discover_cropped_images(config.cropped_dir)
    if not image_paths:
        print("No cropped images found. Creating sample synthetic paths for dry run.")
        image_paths = [Path("synthetic_1027016499237547.jpg"), Path("synthetic_1133313717757155.jpg")]

    detections: List[DetectionRecord] = []
    for image_path in image_paths:
        image_id = parse_image_id_from_filename(image_path)
        metadata_row = find_metadata_row(metadata_df, image_id)

        if config.use_mock_mode:
            det = mock_vision_inference(image_path, metadata_row, config)
        else:
            det = gemini_vision_inference_placeholder(image_path, metadata_row, config)

        validate_detection(det)
        detections.append(det)

    matched: List[MatchedDetection] = []
    merged_features: List[Dict[str, Any]] = []

    for det in detections:
        sign_idx, sign_type, sign_dist = nearest_sign_match(det, all_sign_features, config.max_sign_match_m)
        road_id, road_snap_dist, road_t, _ = nearest_road_snap(det.latitude, det.longitude, roads_features, config.max_road_snap_m)
        school_dist, suppliers = nearest_school(det.latitude, det.longitude, school_features, config.max_school_match_m)

        row = MatchedDetection(
            detection=det,
            matched_sign_index=sign_idx,
            matched_sign_type=sign_type,
            sign_distance_m=sign_dist,
            matched_road_id=road_id,
            road_snap_distance_m=road_snap_dist,
            road_position_t=road_t,
            nearest_school_distance_m=school_dist,
            nearest_school_suppliers=suppliers,
            zone_role="candidate_start_or_end" if road_id else "unknown",
        )
        matched.append(row)

        if det.latitude is not None and det.longitude is not None:
            merged_features.append(
                {
                    "type": "Feature",
                    "geometry": {"type": "Point", "coordinates": [det.longitude, det.latitude]},
                    "properties": {
                        "image_path": det.image_path,
                        "image_id": det.image_id,
                        "sign_type": det.sign_type,
                        "speed_limit_kmh": det.speed_limit_kmh,
                        "time_modifier": asdict(det.time_modifier),
                        "confidence": det.confidence,
                        "notes": det.notes,
                        "model_name": det.model_name,
                        "matched_sign_index": row.matched_sign_index,
                        "matched_sign_type": row.matched_sign_type,
                        "sign_distance_m": row.sign_distance_m,
                        "matched_road_id": row.matched_road_id,
                        "road_snap_distance_m": row.road_snap_distance_m,
                        "road_position_t": row.road_position_t,
                        "nearest_school_distance_m": row.nearest_school_distance_m,
                        "nearest_school_suppliers": row.nearest_school_suppliers,
                        "zone_role": row.zone_role,
                    },
                }
            )

    validated_decisions = multi_step_validate_detections(detections, matched, config)
    line_features, poly_features = infer_zone_features(matched, config)

    with config.image_detections_jsonl.open("w", encoding="utf-8") as f:
        for det in detections:
            row = asdict(det)
            row["time_modifier"] = asdict(det.time_modifier)
            f.write(json.dumps(row, ensure_ascii=True) + "\n")

    export_validated_decisions(config.validated_decisions_jsonl, validated_decisions)
    safe_json_dump(config.merged_detections_geojson, make_feature_collection(merged_features))
    safe_json_dump(config.inferred_lines_geojson, make_feature_collection(line_features))
    safe_json_dump(config.inferred_polygons_geojson, make_feature_collection(poly_features))

    return {
        "n_images": len(image_paths),
        "n_detections": len(detections),
        "n_validated_decisions": len(validated_decisions),
        "n_matched_points": len(merged_features),
        "n_zone_lines": len(line_features),
        "n_zone_polygons": len(poly_features),
        "outputs": {
            "image_detections_jsonl": str(config.image_detections_jsonl),
            "validated_decisions_jsonl": str(config.validated_decisions_jsonl),
            "merged_detections_geojson": str(config.merged_detections_geojson),
            "inferred_lines_geojson": str(config.inferred_lines_geojson),
            "inferred_polygons_geojson": str(config.inferred_polygons_geojson),
        },
    }

## 7. Run a Minimal End-to-End Example

In [8]:
school_region_summary = run_gemini_school_region_pipeline(CONFIG)
pd.DataFrame([school_region_summary])

,input_path,output_geojson,output_response,selected_road_count,selected_supporting_count,confidence,school_label
0,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,K:\Tech\Hackathon\Here-Berlin-2026\Sign_Detect...,30,8,0.95,School at Dessauerstraße


In [9]:
if CONFIG.school_region_geojson.exists():
    school_region_data = json.loads(CONFIG.school_region_geojson.read_text(encoding="utf-8"))
    feature_rows = []
    for feature in school_region_data.get("features", []):
        feature_rows.append({
            "output_role": feature.get("properties", {}).get("output_role"),
            "geometry_type": feature.get("geometry", {}).get("type"),
            "source_feature_index": feature.get("properties", {}).get("source_feature_index"),
        })
    pd.DataFrame(feature_rows)
else:
    print(f"Missing expected output: {CONFIG.school_region_geojson}")

## 8. Write Basic Unit Tests in-Notebook

In [19]:
# Basic function tests
assert extract_speed_limit_kmh("SpeedLimit2V30") == 30
assert extract_speed_limit_kmh("EndSpeedLimit") is None

mod = parse_time_modifier("SupplementalTimeModifier")
assert mod.has_modifier is True
assert mod.type == "time_range"

assert classify_sign_type("SpeedLimit2V30", "SchoolZone3", "School") == "School"
assert classify_sign_type(None, None, None) == "General"

# Validation stage smoke test
sample_det = DetectionRecord(
    image_path="x.jpg",
    image_id="1",
    latitude=52.4,
    longitude=13.3,
    sign_type="School",
    speed_limit_kmh=30,
    time_modifier=TimeModifier(True, "time_range", "07:00-17:00"),
    confidence=0.9,
    notes="ok",
    model_name="mock",
)
validate_detection(sample_det)

sample_match = MatchedDetection(
    detection=sample_det,
    matched_sign_index=0,
    matched_sign_type="SpeedLimit2V30",
    sign_distance_m=12.0,
    matched_road_id="road-1",
    road_snap_distance_m=5.0,
    road_position_t=0.2,
    nearest_school_distance_m=40.0,
    nearest_school_suppliers="School",
)
validated = validate_zone_evidence(
    ZoneEvidence(
        road_id="road-1",
        sign_type="School",
        speed_limit_kmh=30,
        cluster_size=3,
        image_count=2,
        school_distance_m=40.0,
        road_span_m=120.0,
        start_position_t=0.2,
        end_position_t=0.7,
        has_conditional_modifier=True,
        conditional_modifier="Mo-Fr",
        supporting_image_ids=["1", "2"],
        supporting_notes=["Conditional signals: Mo-Fr"],
        confidence=0.8,
    ),
    CONFIG,
)
assert validated.is_school_zone is True
assert validated.is_conditional is True
assert validated.speed_limit_kmh == 30

print("All in-notebook tests passed.")

All in-notebook tests passed.


## Notes

Tests are lightweight smoke checks to keep the notebook fast in hackathon iteration cycles.